# 06. Age Prediction: ONT Cross-Reference Evaluation

Apply every age reference (built in 05) to every read type using the compact maximum-likelihood estimator.  Produces an N_ref x N_read MAE / correlation / bias matrix for the ONT-only cohort test set.

**Inputs:**
- `results/age_references_v2/reference_index.csv` (from 05)
- Bulk and cell-sorted `.beta` files for ONT test-set samples

**Outputs** (`results/age_references_v2/cross_reference_ont_pooled_testonly_agegrid_0_100/`):
- `summary_df.csv`, `mae_matrix.csv`, `corr_matrix.csv`, `nok_matrix.csv`
- Heatmap and scatter PDFs/PNGs

---

### Config, GCS helpers, reference loader, and prediction engine

In [ ]:
import os
import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.cloud import storage
import subprocess

# ============================================================
# 0) CONFIG
# ============================================================
WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
assert WORKSPACE_BUCKET.startswith("gs://")
BUCKET_NAME = WORKSPACE_BUCKET.replace("gs://", "")

OUTPUT_ROOT = "results/age_references_v2"
REF_OUT_PREFIX = f"{WORKSPACE_BUCKET}/{OUTPUT_ROOT}"
REFERENCE_INDEX_GS = f"{REF_OUT_PREFIX}/reference_index.csv"

K_INFER = 1000

CELL_TYPES = [
    "Myeloid",
    "Lymphoid",
    "T_Cell",
    "B_Cell",
    "NK_Cell",
    "Monocyte",
    "Granulocyte",
]

# We will use a special label for ONT-pooled bulk
BULK_ONT_LABEL = "bulk_ONT_pooled"
ALL_TYPES = [BULK_ONT_LABEL] + CELL_TYPES

REFERENCE_GROUPS = {
    "Myeloid":     [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Myeloid"],
    "Lymphoid":    [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Lymphoid"],
    "T_Cell":      [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/T_Cell"],
    "B_Cell":      [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/B_Cell"],
    "NK_Cell":     [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/NK_Cell"],
    "Monocyte":    [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Monocyte"],
    "Granulocyte": [f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Granulocyte"],
}

# Restrict ONT samples to these bulk output folders
ONT_BULK_BATCHES = {
    "bcm_ont",
    "jhu_ont",
}

client = storage.Client()

# ============================================================
# 1) GCS / IO HELPERS
# ============================================================
def gs_to_bucket_blob(gs_path: str):
    assert gs_path.startswith("gs://"), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split("/", 1)
    return bkt, obj

def download_text_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        return None
    return blob.download_as_text()

def read_csv_from_gcs_text(gs_path: str) -> pd.DataFrame:
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return pd.read_csv(io.StringIO(txt))

def load_json_from_gcs(gs_path: str):
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return json.loads(txt)

def load_npy_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    data = blob.download_as_bytes()
    return np.load(io.BytesIO(data), allow_pickle=False)

def load_beta_pairs_from_gcs(gs_path: str, dtype=np.uint8, min_bytes=1000) -> np.ndarray:
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    data = blob.download_as_bytes()

    if data is None or len(data) < min_bytes:
        raise ValueError(f"Empty/tiny beta file: {gs_path} bytes={0 if data is None else len(data)}")

    arr = np.frombuffer(data, dtype=dtype)
    if arr.size % 2:
        arr = arr[:-1]

    pairs = arr.reshape(-1, 2)
    if pairs.shape[0] == 0:
        raise ValueError(f"Parsed 0 CpGs from beta: {gs_path} bytes={len(data)}")
    return pairs

# ============================================================
# 2) REFERENCE / INFERENCE HELPERS
# ============================================================
def load_reference_index():
    txt = download_text_from_gcs(REFERENCE_INDEX_GS)
    if txt is None or txt.strip() == "":
        raise FileNotFoundError(f"Missing or empty reference index: {REFERENCE_INDEX_GS}")
    return pd.read_csv(io.StringIO(txt))

def load_compact_reference(
    compact_prefix: str,
    age_grid_min: float = 0.0,
    age_grid_max: float = 100.0,
    age_grid_step: float = 0.1,
):
    ref = {
        "eligible_idx": load_npy_from_gcs(f"{compact_prefix}eligible_idx.npy").astype(np.int32),
        "intercept_eligible": load_npy_from_gcs(f"{compact_prefix}intercept_eligible.npy").astype(np.float32),
        "coef_eligible": load_npy_from_gcs(f"{compact_prefix}coef_eligible.npy").astype(np.float32),
        "t_stat_eligible": load_npy_from_gcs(f"{compact_prefix}t_stat_eligible.npy").astype(np.float32),
    }

    # Force inference to search over 0 to 100 years,
    # instead of the saved cohort-limited reference grid.
    ref["age_grid"] = np.arange(
        age_grid_min,
        age_grid_max + age_grid_step / 2,
        age_grid_step,
        dtype=np.float32,
    )

    scalars = load_json_from_gcs(f"{compact_prefix}scalars.json")
    ref["EPS"] = float(scalars["EPS"])
    ref["CAP"] = float(scalars["CAP"])

    return ref

def ll_curve(m_use, t_use, b0, b1, age_grid, eps):
    u_use = t_use - m_use
    out = np.empty(len(age_grid), dtype=np.float32)
    for i, a in enumerate(age_grid):
        p = b0 + b1 * float(a)
        p = np.clip(p, eps, 1.0 - eps)
        out[i] = np.sum(m_use * np.log(p) + u_use * np.log(1.0 - p))
    return out

def min_eval_sites_for_K(K: int) -> int:
    return min(10, K)

def predict_from_compact_reference(gs_beta_path: str, compact_ref: dict, K: int = 1000):
    pairs = load_beta_pairs_from_gcs(gs_beta_path, dtype=np.uint8)

    eligible_idx = compact_ref["eligible_idx"]
    if eligible_idx.size == 0:
        return np.nan, 0, "no_eligible_sites", None

    m = pairs[:, 0].astype(np.float32)
    t = pairs[:, 1].astype(np.float32)

    available_mask = t[eligible_idx] > 0
    if available_mask.sum() == 0:
        return np.nan, 0, "no_overlap", None

    idx_local = np.where(available_mask)[0]
    order = np.argsort(compact_ref["t_stat_eligible"][idx_local])[::-1]
    idx_local = idx_local[order][:min(K, len(idx_local))]

    if len(idx_local) < min_eval_sites_for_K(K):
        return np.nan, int(len(idx_local)), "too_few_sites", None

    chosen_global_idx = eligible_idx[idx_local]

    m_raw = m[chosen_global_idx]
    t_raw = t[chosen_global_idx]

    t_use = np.minimum(t_raw, compact_ref["CAP"])
    scale = np.divide(
        t_use,
        t_raw,
        out=np.ones_like(t_raw, dtype=np.float32),
        where=t_raw > 0
    )
    m_use = m_raw * scale

    b0 = compact_ref["intercept_eligible"][idx_local]
    b1 = compact_ref["coef_eligible"][idx_local]
    age_grid = compact_ref["age_grid"]

    ll = ll_curve(m_use, t_use, b0, b1, age_grid, compact_ref["EPS"])
    best = float(age_grid[np.argmax(ll)])
    return best, int(len(idx_local)), "ok", ll

def evaluate_manifest_with_compact_reference(manifest_df: pd.DataFrame, beta_col: str, compact_prefix: str, K: int = 1000):
    compact_ref = load_compact_reference(compact_prefix)

    rows = []
    for row in tqdm(manifest_df.itertuples(index=False), total=len(manifest_df), desc=f"Infer {beta_col} K={K}"):
        try:
            pred, nsites, status, _ = predict_from_compact_reference(
                getattr(row, beta_col),
                compact_ref,
                K=K
            )
        except Exception as e:
            pred, nsites, status = np.nan, 0, f"error:{type(e).__name__}"

        rows.append({
            "person_id": str(row.person_id),
            "true_age": float(row.Age),
            "pred_age": pred,
            "n_sites_used": nsites,
            "status": status
        })

    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()

    if len(ok) == 0:
        summary = {
            "n_ok": 0,
            "MAE": np.nan,
            "bias": np.nan,
            "corr": np.nan,
            "pred_std": np.nan,
        }
    else:
        err = ok["pred_age"] - ok["true_age"]
        summary = {
            "n_ok": int(len(ok)),
            "MAE": float(np.mean(np.abs(err))),
            "bias": float(np.mean(err)),
            "corr": float(np.corrcoef(ok["pred_age"], ok["true_age"])[0, 1]) if len(ok) > 1 else np.nan,
            "pred_std": float(np.std(ok["pred_age"])) if len(ok) > 1 else np.nan,
        }

    return df, summary

# ============================================================
# 3) READ DISCOVERY
# ============================================================
def parse_celltype_filename(fname: str, celltype: str):
    suffix = f"_{celltype}.beta"
    if not fname.endswith(suffix):
        return None
    core = fname[:-len(suffix)]
    if "__" not in core:
        return None
    batch_token, person_id = core.split("__", 1)
    if len(person_id) == 0:
        return None
    return {"batch_token": batch_token, "person_id": str(person_id)}

def discover_celltype_manifest_for_people_ont(celltype: str, people_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    people_df = people_df.copy()
    people_df["person_id"] = people_df["person_id"].astype(str)
    people_df["Age"] = pd.to_numeric(people_df["Age"], errors="coerce")

    for cell_prefix_gs in REFERENCE_GROUPS[celltype]:
        prefix_no_gs = cell_prefix_gs.replace(f"gs://{BUCKET_NAME}/", "")
        print(f"Scanning {celltype}: {cell_prefix_gs}")

        for blob in client.list_blobs(BUCKET_NAME, prefix=prefix_no_gs):
            if not blob.name.endswith(".beta"):
                continue

            fname = blob.name.split("/")[-1]
            parsed = parse_celltype_filename(fname, celltype)
            if parsed is None:
                continue

            if parsed["batch_token"] not in ONT_BULK_BATCHES:
                continue

            rows.append({
                "person_id": str(parsed["person_id"]),
                "batch_token": parsed["batch_token"],
                "celltype": celltype,
                "celltype_beta_path": f"gs://{BUCKET_NAME}/{blob.name}",
                "size_bytes": int(blob.size or 0),
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    df = df[df["size_bytes"] >= 1000].copy()
    df["person_id"] = df["person_id"].astype(str)

    df = people_df.merge(
        df[["person_id", "celltype", "celltype_beta_path", "batch_token"]],
        on="person_id",
        how="inner"
    ).copy()

    df = df.sort_values(["person_id", "celltype_beta_path"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["person_id"]).reset_index(drop=True)
    return df

def get_reference_row(ref_index: pd.DataFrame, ref_type: str):
    if ref_type == BULK_ONT_LABEL:
        hit = ref_index[
            (ref_index["modality"] == "bulk") &
            (ref_index["celltype"] == "bulk") &
            (ref_index["group_level"] == "technology") &
            (ref_index["group_name"] == "ONT")
        ].copy()
    else:
        hit = ref_index[
            (ref_index["modality"] == "celltype") &
            (ref_index["celltype"] == ref_type) &
            (ref_index["group_level"] == "technology") &
            (ref_index["group_name"] == "ONT")
        ].copy()

    if len(hit) == 0:
        raise ValueError(f"No ONT pooled reference found for ref_type={ref_type}")
    return hit.iloc[0]

def get_ont_bulk_test_manifest(ref_index: pd.DataFrame):
    bulk_ont_ref = get_reference_row(ref_index, BULK_ONT_LABEL)
    bulk_test = read_csv_from_gcs_text(bulk_ont_ref["test_manifest_gs"])
    bulk_test["person_id"] = bulk_test["person_id"].astype(str)
    bulk_test["Age"] = pd.to_numeric(bulk_test["Age"], errors="coerce")

    # keep only ONT batches if source_prefix exists
    if "source_prefix" in bulk_test.columns:
        bulk_test["batch_token"] = bulk_test["source_prefix"].astype(str).str.split("/").str[-1]
        bulk_test = bulk_test[bulk_test["batch_token"].isin(ONT_BULK_BATCHES)].copy()

    return bulk_test

def get_read_manifest(read_type: str, people_df: pd.DataFrame, bulk_test_manifest: pd.DataFrame):
    if read_type == BULK_ONT_LABEL:
        out = bulk_test_manifest.copy()
        out["person_id"] = out["person_id"].astype(str)
        out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
        return out[["person_id", "Age", "bulk_beta_path"]].copy()

    ct_manifest = discover_celltype_manifest_for_people_ont(
        read_type,
        people_df[["person_id", "Age"]].copy()
    )
    return ct_manifest[["person_id", "Age", "celltype_beta_path"]].copy()

# ============================================================
# 4) LOAD ONT TEST PEOPLE
# ============================================================
ref_index = load_reference_index()

bulk_test = get_ont_bulk_test_manifest(ref_index)
people_df = bulk_test[["person_id", "Age"]].drop_duplicates().copy()

print("ONT bulk TEST rows:", len(bulk_test))
print("ONT bulk TEST unique people:", bulk_test["person_id"].nunique())
display(bulk_test.head())

# ============================================================
# 5) BUILD READ MANIFESTS FOR ALL ONT TYPES
# ============================================================
read_manifests = {}
read_beta_col = {}

for read_type in ALL_TYPES:
    print("\n" + "=" * 90)
    print(f"Preparing ONT manifest for read_type = {read_type}")

    manifest_df = get_read_manifest(read_type, people_df, bulk_test)
    read_manifests[read_type] = manifest_df
    read_beta_col[read_type] = "bulk_beta_path" if read_type == BULK_ONT_LABEL else "celltype_beta_path"

    print(f"{read_type}: {len(manifest_df)} rows")
    if len(manifest_df) > 0:
        display(manifest_df.head())

# ============================================================
# 6) RUN FULL ONT-ONLY 8 x 8 EVALUATION
# ============================================================
all_eval_results = {}
summary_rows = []

for ref_type in ALL_TYPES:
    print("\n" + "#" * 110)
    print(f"REFERENCE = {ref_type}")
    ref_row = get_reference_row(ref_index, ref_type)
    compact_prefix = ref_row["compact_prefix"]

    for read_type in ALL_TYPES:
        print("\n" + "-" * 90)
        print(f"Applying ONT ref={ref_type} to ONT reads={read_type}")

        manifest_df = read_manifests[read_type]
        beta_col = read_beta_col[read_type]

        if len(manifest_df) == 0:
            summary = {
                "reference": ref_type,
                "read_type": read_type,
                "n_manifest": 0,
                "n_ok": 0,
                "MAE": np.nan,
                "bias": np.nan,
                "corr": np.nan,
                "pred_std": np.nan,
            }
            summary_rows.append(summary)
            continue

        eval_df, summary = evaluate_manifest_with_compact_reference(
            manifest_df=manifest_df,
            beta_col=beta_col,
            compact_prefix=compact_prefix,
            K=K_INFER
        )

        summary["reference"] = ref_type
        summary["read_type"] = read_type
        summary["n_manifest"] = int(len(manifest_df))

        all_eval_results[(ref_type, read_type)] = eval_df
        summary_rows.append(summary)

        display(pd.DataFrame([summary]))

summary_df = pd.DataFrame(summary_rows)
print("\nFull ONT-only summary:")
display(summary_df)

# ============================================================
# 7) PIVOT TABLES
# ============================================================
mae_mat = summary_df.pivot(index="reference", columns="read_type", values="MAE").reindex(index=ALL_TYPES, columns=ALL_TYPES)
corr_mat = summary_df.pivot(index="reference", columns="read_type", values="corr").reindex(index=ALL_TYPES, columns=ALL_TYPES)
nok_mat = summary_df.pivot(index="reference", columns="read_type", values="n_ok").reindex(index=ALL_TYPES, columns=ALL_TYPES)

print("\nMAE matrix")
display(mae_mat)

print("\nCorrelation matrix")
display(corr_mat)

print("\nN ok matrix")
display(nok_mat)

# ============================================================
# 8) SAVE TABLES TO GCS
# ============================================================
OUT_GCS = f"{WORKSPACE_BUCKET}/{OUTPUT_ROOT}/cross_reference_ont_pooled_testonly_agegrid_0_100"
os.makedirs("/tmp/cross_ref_ont_tmp", exist_ok=True)

for name, df in [
    ("summary_df.csv", summary_df),
    ("mae_matrix.csv", mae_mat.reset_index()),
    ("corr_matrix.csv", corr_mat.reset_index()),
    ("nok_matrix.csv", nok_mat.reset_index()),
]:
    local = f"/tmp/cross_ref_ont_tmp/{name}"
    df.to_csv(local, index=False)
    r = subprocess.run(f"gsutil cp {local} {OUT_GCS}/{name}", shell=True, capture_output=True)
    print(("" if r.returncode == 0 else ""), f"{OUT_GCS}/{name}")

# ============================================================
# 9) QUICK HEATMAPS
# ============================================================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def draw_heatmap(mat, title, cmap="viridis", fmt=".2f", vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")
    arr = mat.values.astype(float)
    im = ax.imshow(arr, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            txt = "NA" if np.isnan(val) else format(val, fmt)
            ax.text(j, i, txt, ha="center", va="center", fontsize=7, color="black")

    ax.set_xlabel("Read type")
    ax.set_ylabel("Reference")
    ax.set_title(title, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    return fig

fig_mae = draw_heatmap(mae_mat, f"ONT-only MAE matrix  |  K={K_INFER}", cmap="magma_r")
plt.show()

fig_corr = draw_heatmap(corr_mat, f"ONT-only correlation matrix  |  K={K_INFER}", cmap="viridis", vmin=0, vmax=1)
plt.show()

fig_nok = draw_heatmap(nok_mat, "ONT-only n_ok matrix", cmap="Blues", fmt=".0f")
plt.show()

for stem, fig in [
    ("heatmap_mae", fig_mae),
    ("heatmap_corr", fig_corr),
    ("heatmap_nok", fig_nok),
]:
    local = f"/tmp/cross_ref_ont_tmp/{stem}.png"
    fig.savefig(local, bbox_inches="tight", facecolor="white")
    r = subprocess.run(f"gsutil cp {local} {OUT_GCS}/{stem}.png", shell=True, capture_output=True)
    print(("" if r.returncode == 0 else ""), f"{OUT_GCS}/{stem}.png")

print(f"\nDone. Outputs written to: {OUT_GCS}/")

### Publication-quality heatmaps (seaborn + clustermap)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import subprocess

# ============================================================
# 9) PUBLICATION-QUALITY PLOTS (UPDATED)
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,  
    "ps.fonttype": 42,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

def draw_publication_heatmap(mat, title, cmap="viridis", fmt=".2f", vmin=None, vmax=None, cbar_label=""):
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
    arr = mat.astype(float)

    sns.heatmap(
        arr,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        square=True,
        linewidths=0.5,
        linecolor='white',
        cbar_kws={"shrink": 0.82, "label": cbar_label},
        # REMOVED "color": "black" so Seaborn auto-inverts text color for readability
        annot_kws={"size": 8, "weight": "normal"}, 
        ax=ax
    )

    ax.set_title(title, fontweight="bold", pad=20)
    ax.set_xlabel("Read type", fontweight="bold", labelpad=10)
    ax.set_ylabel("Reference", fontweight="bold", labelpad=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.tick_params(left=False, bottom=False)

    plt.tight_layout()
    return fig

# ------------------------------------------------------------
# Plot A: Fixed MAE Heatmap
# ------------------------------------------------------------
fig_mae = draw_publication_heatmap(
    mae_mat, 
    title=f"ONT-only MAE Matrix  |  K={K_INFER}", 
    cmap="rocket_r", 
    vmin=0, 
    cbar_label="Mean Absolute Error (Years)"
)
plt.show()

# ------------------------------------------------------------
# Plot B: Hierarchical Clustermap (Biological Similarity)
# ------------------------------------------------------------
def draw_clustermap(mat, title, cmap="rocket_r", fmt=".2f"):
    arr = mat.astype(float)
    
    cg = sns.clustermap(
        arr,
        cmap=cmap,
        annot=True,
        fmt=fmt,
        linewidths=0.5,
        linecolor='white',
        figsize=(8.5, 8.5),
        cbar_kws={"label": "Mean Absolute Error (Years)"},
        annot_kws={"size": 8, "weight": "normal"}
    )
    
    cg.ax_heatmap.set_xlabel("Read type", fontweight="bold", labelpad=10)
    cg.ax_heatmap.set_ylabel("Reference", fontweight="bold", labelpad=10)
    cg.ax_heatmap.set_xticklabels(cg.ax_heatmap.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    cg.ax_heatmap.set_yticklabels(cg.ax_heatmap.get_yticklabels(), rotation=0)
    cg.fig.suptitle(title, fontweight="bold", y=1.02)
    
    return cg.fig

fig_cluster = draw_clustermap(mae_mat, f"Hierarchical Clustering of MAE Profiles | K={K_INFER}")
plt.show()

# ------------------------------------------------------------
# Plot C: Matched vs. Mismatched Cell Type Performance
# ------------------------------------------------------------
def draw_matched_vs_mismatched(summary_df):
    # Filter out the bulk references for a clean cell-type specific comparison
    ct_df = summary_df[
        (summary_df["reference"] != "bulk_ONT_pooled") & 
        (summary_df["read_type"] != "bulk_ONT_pooled")
    ].copy()
    
    # Tag whether the reference matches the read type
    ct_df["Status"] = np.where(ct_df["reference"] == ct_df["read_type"], "Matched", "Mismatched")
    
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    
    # Draw a boxplot with individual points overlaid
    sns.boxplot(
        data=ct_df, 
        x="Status", 
        y="MAE", 
        order=["Matched", "Mismatched"],
        palette=["#4C72B0", "#C44E52"], 
        width=0.4, 
        fliersize=0, 
        ax=ax
    )
    sns.stripplot(
        data=ct_df, 
        x="Status", 
        y="MAE", 
        order=["Matched", "Mismatched"],
        color="black", 
        alpha=0.6, 
        jitter=True, 
        ax=ax
    )
    
    ax.set_title("Clock Performance: Matched vs. Mismatched Lineages", fontweight="bold", pad=15)
    ax.set_ylabel("Mean Absolute Error (Years)", fontweight="bold")
    ax.set_xlabel("Reference vs. Read Type", fontweight="bold")
    
    # Use a solid grid line for the y-axis to guide the eye
    ax.grid(axis='y', color='gray', alpha=0.2, linestyle='-')
    ax.set_axisbelow(True)
    
    # Clean up outer spines
    sns.despine()
    plt.tight_layout()
    
    return fig

fig_matched = draw_matched_vs_mismatched(summary_df)
plt.show()

# ------------------------------------------------------------
# Save new plots to GCS
# ------------------------------------------------------------
for stem, fig in [
    ("heatmap_mae_pub", fig_mae),
    ("clustermap_mae_pub", fig_cluster),
    ("matched_vs_mismatched_pub", fig_matched),
]:
    local_pdf = f"/tmp/cross_ref_ont_tmp/{stem}.pdf"
    fig.savefig(local_pdf, bbox_inches="tight", facecolor="white", format="pdf")
    subprocess.run(f"gsutil cp {local_pdf} {OUT_GCS}/{stem}.pdf", shell=True)

### ONT cohort visualizations: density scatter, EAD violins, age-stratified bias

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import subprocess

# ============================================================
# 11) ONT COHORT VISUALIZATIONS (ALL ROW-LEVEL INFERENCES)
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,  
    "ps.fonttype": 42,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# 1. Compile row-level data from the dictionary created in Step 6
cohort_rows = []
for (ref_type, read_type), df in all_eval_results.items():
    if len(df) > 0:
        df_copy = df.copy()
        df_copy["reference"] = ref_type
        df_copy["read_type"] = read_type
        cohort_rows.append(df_copy)

cohort_df = pd.concat(cohort_rows, ignore_index=True)

# Filter only successful inferences
cohort_df = cohort_df[cohort_df["status"] == "ok"].copy()

# Calculate Epigenetic Age Deviation (Residual Error)
cohort_df["EAD"] = cohort_df["pred_age"] - cohort_df["true_age"]
cohort_df["abs_error"] = cohort_df["EAD"].abs()

# Isolate the Matched inferences (Ref == Read) for a clean biological baseline
# With 65 people and 8 read types, this yields 520 matched data points
matched_df = cohort_df[cohort_df["reference"] == cohort_df["read_type"]].copy()

# ------------------------------------------------------------
# Plot G: ONT Density Scatter (True vs Pred Age) for Matched Cells
# ------------------------------------------------------------
def draw_ont_density_scatter(df):
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
    
    x = df["true_age"].values
    y = df["pred_age"].values
    xy = np.vstack([x, y])
    z = gaussian_kde(xy)(xy)
    
    idx = z.argsort()
    x, y, z = x[idx], y[idx], z[idx]
    
    sc = ax.scatter(x, y, c=z, s=20, cmap="mako_r", edgecolor="none", alpha=0.8)
    
    min_val = min(x.min(), y.min()) - 5
    max_val = max(x.max(), y.max()) + 5
    ax.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", linewidth=1.5, label="y = x (Perfect prediction)")
    
    corr = np.corrcoef(x, y)[0, 1]
    mae = np.mean(np.abs(x - y))
    textstr = f'$r = {corr:.3f}$\n$MAE = {mae:.2f} yrs$'
    props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11, verticalalignment='top', bbox=props)
    
    ax.set_title("ONT Matched: Predicted vs. Chronological Age", fontweight="bold", pad=15)
    ax.set_xlabel("Chronological Age (Years)", fontweight="bold")
    ax.set_ylabel("Predicted Age (Years)", fontweight="bold")
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    
    plt.legend(loc="lower right", frameon=False)
    sns.despine()
    plt.tight_layout()
    return fig

fig_density_ont = draw_ont_density_scatter(matched_df)
plt.show()

# ------------------------------------------------------------
# Plot H: ONT Age Deviation Violin Plots (By Cell Type)
# ------------------------------------------------------------
def draw_ont_ead_violins(df):
    fig, ax = plt.subplots(figsize=(9, 5), facecolor="white")
    
    order = df.groupby("read_type")["EAD"].median().sort_values().index
    
    sns.violinplot(
        data=df,
        x="read_type",
        y="EAD",
        order=order,
        palette="viridis",
        inner="quartile",
        linewidth=1,
        ax=ax
    )
    
    ax.axhline(0, color="black", linestyle="-", linewidth=1, alpha=0.5)
    
    ax.set_title("ONT Epigenetic Age Deviation by Lineage", fontweight="bold", pad=15)
    ax.set_xlabel("ONT Lineage (Matched Reference)", fontweight="bold")
    ax.set_ylabel("Age Deviation (Years)", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    
    ax.grid(axis='y', color='gray', alpha=0.2, linestyle='solid')
    ax.set_axisbelow(True)
    
    sns.despine()
    plt.tight_layout()
    return fig

fig_violins_ont = draw_ont_ead_violins(matched_df)
plt.show()

# ------------------------------------------------------------
# Plot I: ONT Age-Stratified Error (Saturation Check)
# ------------------------------------------------------------
def draw_ont_age_stratified_error(df):
    fig, ax = plt.subplots(figsize=(7, 5), facecolor="white")
    
    min_age = int(np.floor(df["true_age"].min() / 10.0)) * 10
    max_age = int(np.ceil(df["true_age"].max() / 10.0)) * 10
    bins = range(min_age, max_age + 10, 10)
    labels = [f"{b}-{b+9}" for b in bins[:-1]]
    
    df_strat = df.copy()
    df_strat["Age_Decade"] = pd.cut(df_strat["true_age"], bins=bins, labels=labels, right=False)
    
    sns.pointplot(
        data=df_strat,
        x="Age_Decade",
        y="EAD",
        errorbar="ci",
        capsize=0.1,
        color="#2C3E50",
        markers="o",
        ax=ax
    )
    
    ax.axhline(0, color="red", linestyle="--", label="Perfect Accuracy")
    
    ax.set_title("ONT Age-Dependent Bias (Clock Saturation)", fontweight="bold", pad=15)
    ax.set_xlabel("Chronological Age Decade", fontweight="bold")
    ax.set_ylabel("Average Bias / EAD (Years)", fontweight="bold")
    
    ax.grid(axis='both', color='gray', alpha=0.2, linestyle='solid')
    ax.set_axisbelow(True)
    
    plt.legend(frameon=False)
    sns.despine()
    plt.tight_layout()
    return fig

fig_strat_ont = draw_ont_age_stratified_error(matched_df)
plt.show()

# ------------------------------------------------------------
# Save ONT Large N plots to GCS
# ------------------------------------------------------------
for stem, fig in [
    ("ont_density_scatter_pub", fig_density_ont),
    ("ont_ead_violins_pub", fig_violins_ont),
    ("ont_age_stratified_error_pub", fig_strat_ont)
]:
    local_pdf = f"/tmp/cross_ref_ont_tmp/{stem}.pdf"
    fig.savefig(local_pdf, bbox_inches="tight", facecolor="white", format="pdf")
    subprocess.run(f"gsutil cp {local_pdf} {OUT_GCS}/{stem}.pdf", shell=True)
    
    local_png = f"/tmp/cross_ref_ont_tmp/{stem}.png"
    fig.savefig(local_png, bbox_inches="tight", facecolor="white", format="png")
    subprocess.run(f"gsutil cp {local_png} {OUT_GCS}/{stem}.png", shell=True)

### Hierarchical clustered panels: Ward linkage over all four metrics

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np

metrics = ["MAE", "bias", "corr", "pred_std"]
titles  = ["MAE (years)", "Bias (years)", "Pearson r", "Pred. SD (years)"]
cmaps   = ["YlOrRd", "RdBu_r", "YlGn", "YlOrRd"]

# pivot — uses summary_df and ALL_TYPES from the ONT notebook
pivot = {
    m: summary_df.pivot(index="reference", columns="read_type", values=m).reindex(
        index=ALL_TYPES, columns=ALL_TYPES
    )
    for m in metrics
}

def get_cluster_order(pivot_dict, metrics, all_types):
    feature_rows = []
    for t in all_types:
        row_feats = []
        for m in metrics:
            vals = pivot_dict[m].loc[t].values.astype(float)
            row_feats.append(vals)
        feature_rows.append(np.concatenate(row_feats))

    feat_mat = np.array(feature_rows)
    col_means = np.nanmean(feat_mat, axis=0)
    inds = np.where(np.isnan(feat_mat))
    feat_mat[inds] = np.take(col_means, inds[1])

    Z = linkage(feat_mat, method="ward", metric="euclidean")
    order = leaves_list(Z)
    return order, Z

order, Z = get_cluster_order(pivot, metrics, ALL_TYPES)
clustered_types = [ALL_TYPES[i] for i in order]
ct_labels_clust = [t.replace("_", " ") for t in clustered_types]

def plot_clustered_panel_ont(metric, title, cmap, filename_stem,
                              order=order, Z=Z,
                              clustered_types=clustered_types,
                              ct_labels_clust=ct_labels_clust):

    TICK_FS  = 9
    LABEL_FS = 10.5
    TITLE_FS = 11
    ANNOT_FS = 9

    fig = plt.figure(figsize=(7.5, 6.5))
    fig.patch.set_facecolor("white")

    gs = gridspec.GridSpec(
        2, 2, figure=fig,
        height_ratios=[0.18, 1],
        width_ratios=[1, 0.18],
        hspace=0.04, wspace=0.04,
    )

    ax_dendro_top   = fig.add_subplot(gs[0, 0])
    ax_dendro_right = fig.add_subplot(gs[1, 1])
    ax_heat         = fig.add_subplot(gs[1, 0])

    mat_full = pivot[metric].reindex(index=ALL_TYPES, columns=ALL_TYPES).values.astype(float)
    mat = mat_full[np.ix_(order, order)]

    if metric == "bias":
        vabs = np.nanmax(np.abs(mat))
        vmin, vmax = -vabs, vabs
    else:
        vmin, vmax = np.nanmin(mat), np.nanmax(mat)

    # ── heatmap ──────────────────────────────────────────────────────────────
    im = ax_heat.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax,
                        aspect="auto", interpolation="nearest")

    for i in range(len(clustered_types)):
        for j in range(len(clustered_types)):
            val = mat[i, j]
            if np.isnan(val):
                txt, color = "—", "#888"
            else:
                txt = f"{val:.2f}"
                rgba = plt.get_cmap(cmap)((val - vmin) / (vmax - vmin + 1e-9))
                lum  = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
                color = "white" if lum < 0.45 else "#1a1a1a"
            ax_heat.text(j, i, txt, ha="center", va="center",
                         fontsize=ANNOT_FS, color=color)

    # diagonal boxes remapped to clustered positions
    for orig_idx in range(len(ALL_TYPES)):
        new_pos = order.tolist().index(orig_idx)
        ax_heat.add_patch(plt.Rectangle(
            (new_pos - 0.5, new_pos - 0.5), 1, 1,
            fill=False, edgecolor="#333", lw=1.2, zorder=3
        ))

    ax_heat.set_xticks(range(len(clustered_types)))
    ax_heat.set_yticks(range(len(clustered_types)))
    ax_heat.set_xticklabels(ct_labels_clust, rotation=40, ha="right", fontsize=TICK_FS)
    ax_heat.set_yticklabels(ct_labels_clust, fontsize=TICK_FS)
    ax_heat.set_xlabel("Read type",  fontsize=LABEL_FS, labelpad=5)
    ax_heat.set_ylabel("Reference",  fontsize=LABEL_FS, labelpad=5)
    ax_dendro_top.set_title(title, fontsize=TITLE_FS, fontweight="medium", pad=8)

    # ── colorbar: placed to the right of the right dendrogram ────────────────
    # get the right-dendrogram axis position to anchor the colorbar next to it
    fig.canvas.draw()  # needed so get_position() is accurate before show()

    pos_right = ax_dendro_right.get_position()
    pos_heat  = ax_heat.get_position()

    cbar_left   = pos_right.x1 + 0.02
    cbar_bottom = pos_heat.y0
    cbar_width  = 0.025
    cbar_height = pos_heat.height

    cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_height])
    cb  = fig.colorbar(im, cax=cax, orientation="vertical")
    cb.ax.tick_params(labelsize=8, length=2)
    cb.outline.set_linewidth(0.5)

    # ── top dendrogram ────────────────────────────────────────────────────────
    dendrogram(Z, ax=ax_dendro_top, labels=None,
               color_threshold=0, above_threshold_color="#5F5E5A",
               orientation="top", no_labels=True)
    ax_dendro_top.axis("off")

    # ── right dendrogram ──────────────────────────────────────────────────────
    dendrogram(Z, ax=ax_dendro_right, labels=None,
               color_threshold=0, above_threshold_color="#5F5E5A",
               orientation="right", no_labels=True)
    ax_dendro_right.axis("off")

    caption = (
        f"Figure. {title} when applying each ONT reference (rows) to each ONT read type (columns). "
        "Rows and columns reordered by Ward hierarchical clustering. "
        "Diagonal boxes mark matched reference–read pairs. "
        f"ONT bulk test set n\u2009=\u2009{bulk_test['person_id'].nunique()} individuals "
        f"(batches: {', '.join(sorted(ONT_BULK_BATCHES))})."
    )
    fig.text(0.5, -0.08, caption, ha="center", fontsize=8.5, color="#5F5E5A",
             wrap=True, transform=fig.transFigure)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)

    plt.savefig(f"/home/jupyter/{filename_stem}.pdf", dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(f"/home/jupyter/{filename_stem}.png", dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved {filename_stem}")
    plt.show()


print("ONT setup done — run each panel cell below independently.")

#### Clustered panel: MAE

In [ ]:
plot_clustered_panel_ont(
    metric="MAE",
    title="MAE (years)",
    cmap="YlOrRd",
    filename_stem="ONT_clustered_MAE",
)

#### Clustered panel: Bias

In [ ]:
plot_clustered_panel_ont(
    metric="bias",
    title="Bias (years)",
    cmap="RdBu_r",
    filename_stem="ONT_clustered_bias",
)

#### Clustered panel: Correlation

In [ ]:
plot_clustered_panel_ont(
    metric="corr",
    title="Pearson r",
    cmap="YlGn",
    filename_stem="ONT_clustered_corr",
)

#### Clustered panel: Pred SD

In [ ]:
plot_clustered_panel_ont(
    metric="pred_std",
    title="Pred. SD (years)",
    cmap="YlOrRd",
    filename_stem="ONT_clustered_pred_std",
)